In [1]:
### Next oken probability
# we will take a look at token probabilities on Nemotron
# temperature shapes the probability landscape, we can use techniques to filter that landscape

In [2]:
# load model

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
revision = "2e43387afd60157064e5bef4e9a583f887c6dfdd"  # your cached snapshot
cache_dir = "/data/hf/hub"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=True,
    revision=revision,
    cache_dir=cache_dir,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda:0",
    revision=revision,
    cache_dir=cache_dir,
    local_files_only=True,
)

print("Loaded tokenizer + model on", model.device)

/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

Loaded tokenizer + model on cuda:0


In [6]:
prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    logits = model(**inputs).logits[0, -1].float()  # next-token logits

def show_topk(logits, temperature=1.0, k=10):
    scaled = logits / max(float(temperature), 1e-8)
    probs = torch.softmax(scaled, dim=-1)
    vals, idxs = torch.topk(probs, k=k)
    print(f"\nTop-{k} next tokens (temperature={temperature}):")
    for p, idx in zip(vals.tolist(), idxs.tolist()):
        token_str = tokenizer.decode([idx])
        print(f"{token_str!r:>12}  p={p:.4f}")

print("Prompt:", prompt)
show_topk(logits, temperature=0.5)
show_topk(logits, temperature=1.0)
show_topk(logits, temperature=1.5)


Prompt: The capital of France is

Top-10 next tokens (temperature=0.5):
    ' Paris'  p=0.9997
       ' **'  p=0.0003
    '\u202f'  p=0.0000
      ' **['  p=0.0000
    ' paris'  p=0.0000
        ' <'  p=0.0000
        ' ['  p=0.0000
   ' Berlin'  p=0.0000
       ' \\'  p=0.0000
     'Paris'  p=0.0000

Top-10 next tokens (temperature=1.0):
    ' Paris'  p=0.9731
       ' **'  p=0.0178
    '\u202f'  p=0.0027
      ' **['  p=0.0011
    ' paris'  p=0.0010
        ' <'  p=0.0006
        ' ['  p=0.0004
   ' Berlin'  p=0.0003
       ' \\'  p=0.0003
     'Paris'  p=0.0003

Top-10 next tokens (temperature=1.5):
    ' Paris'  p=0.7456
       ' **'  p=0.0518
    '\u202f'  p=0.0148
      ' **['  p=0.0083
    ' paris'  p=0.0076
        ' <'  p=0.0055
        ' ['  p=0.0041
   ' Berlin'  p=0.0035
       ' \\'  p=0.0033
     'Paris'  p=0.0030
